# Day 4 · Version A: rules and templates (the honest baseline)

**First-principles question:** How far can plain code go before you need a model?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import pandas as pd
from lab.acme import INTENTS, first_name, load_cases, load_customers, outage_for, save_results

cases = load_cases()
customers = load_customers()
print(len(cases), "cases")

## 1 · A keyword classifier
No model. Just words you choose.

In [ ]:
KEYWORDS = {
    "billing": ["bill", "charge", "refund", "payment", "paid", "fee"],
    "outage": ["down", "not working", "no internet", "band hai"],
    # TODO: add keyword lists for "speed", "cancel", "upgrade" and "relocation" (include Hinglish words!)
    #       and improve the two lists above when you see failures
}

def classify(message):
    """The intent whose keywords appear most often in the message, or "other"."""
    text = message.lower()
    # TODO: count, for each intent, how many of its keywords appear in `text`;
    #       return the intent with the highest count, or "other" if every count is 0
    return "other"

classify("My internet is not working since morning")

## 2 · Templates
One reply per intent. `{first_name}`, `{area}` and `{ticket}` are filled in for you.

In [ ]:
TEMPLATES = {
    "billing": "Hi {first_name}, thanks for writing. Bills are due by the 10th, and a ₹100 late fee is added after "
               "that. You can see every bill in the Acme app under Billing.",
    "outage": "Hi {first_name}, there is an outage in {area} (ticket {ticket}). Our engineers are working on it and "
              "will update you by email and in the Acme app.",
    # TODO: write templates for "speed", "cancel", "upgrade", "relocation" and "other"
    #       (use data/acme/kb.md for the facts: 30 days notice, ₹1,499, 48 hours, restart the router…)
}

def version_a(case):
    customer = customers[case["customer_id"]]
    outage = outage_for(customer)
    intent = classify(case["message"])
    fields = {"first_name": first_name(customer), "area": customer["area"],
              "ticket": outage["ticket"] if outage else "not available"}
    template = TEMPLATES.get(intent, TEMPLATES.get("other", "Hi {first_name}, thanks for writing to us."))
    return {"intent": intent, "reply": template.format(**fields), "needs_human": False}

version_a(cases[0])

## 3 · Run version A on every case

In [ ]:
rows = []
for case in cases:
    out = version_a(case)
    rows.append(dict(out, id=case["id"], tokens_in=0, tokens_out=0, seconds=0.0))
save_results("A", rows)

df = pd.DataFrame({"id": [c["id"] for c in cases], "expected": [c["expected_intent"] for c in cases],
                   "predicted": [r["intent"] for r in rows]})
accuracy = (df.expected == df.predicted).mean()
print(f"Intent accuracy: {accuracy:.0%} on {len(df)} cases")
pd.crosstab(df.expected, df.predicted, margins=True)

## 4 · Five failures
Read them. Then write one line per failure in the markdown cell below.

In [ ]:
wrong = df[df.expected != df.predicted]
for _, row in wrong.head(5).iterrows():
    case = next(c for c in cases if c["id"] == row.id)
    print(f"{row.id}: expected {row.expected}, got {row.predicted}\n   {case['message']}\n")

**Why the rules failed** (one line each):

1. …
2. …
3. …
4. …
5. …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert classify("My bill has an extra charge") == "billing"
assert classify("internet is down since morning") == "outage"
assert classify("hello there") == "other", 'No keyword at all should give "other"'
missing = [i for i in INTENTS if i not in TEMPLATES]
assert not missing, f"Write templates for: {missing}"
assert len(rows) == len(cases) and 0 <= accuracy <= 1
assert (ROOT / "data" / "results" / "A.jsonl").exists()
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")